In [ ]:
from pathlib import Path
import os, json, hashlib, platform
import numpy as np
import pandas as pd

PROJECT = Path.cwd().resolve()
ROOT = Path(os.environ.get("PINPAD_DATA_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate"))
OUTPUT = Path(os.environ.get("PINPAD_OUTPUT_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/PINPAD_Compact_Results"))
N_BUTTONS, N_PER_BUTTON = 12, 200
RESPONSE_LENGTH = 3624
INPUT_FORMAT = "magnitude"
NORMALIZATION = "source_global_zscore"
DRIFT_MODE = "none"
RUN_SEEDS = [42,43,44,45,46]
SOURCE = [(c,"R1") for c in range(1,6)]
ADAPT_CC = [(c,"R1") for c in range(6,11)]
ADAPT_CR = [(c,"R2") for c in range(1,6)]
ADAPT_CCR = [(c,"R2") for c in range(11,16)]
ADAPT = {"Source":SOURCE,"CC":ADAPT_CC,"CR":ADAPT_CR,"CCR":ADAPT_CCR}
EXPECTED = set(sum(ADAPT.values(),[]))
COLLECTION_MAP = ROOT/"collection_paths.csv"


def collection_paths():
    mapping = {}
    if COLLECTION_MAP.is_file():
        table = pd.read_csv(COLLECTION_MAP,dtype=str,keep_default_na=False)
        if not {"card","reader","directory"} <= set(table):
            raise ValueError("collection_paths.csv requires card,reader,directory")
        for row in table.itertuples(index=False):
            key = (int(row.card),row.reader.strip().upper())
            if key in mapping or not row.directory.strip():
                raise ValueError(f"Duplicate or empty collection mapping: {key}")
            path = Path(row.directory).expanduser()
            mapping[key] = (path if path.is_absolute() else ROOT/path).resolve()
    else:
        for card,reader in sorted(EXPECTED):
            candidates = {p.resolve() for p in (ROOT/reader/f"card{card}",
                ROOT/reader.lower()/f"card{card}",ROOT/f"card{card}"/reader,
                ROOT/f"card{card}"/reader.lower()) if p.is_dir()}
            if len(candidates)>1: raise ValueError(f"Ambiguous physical collection: {card}, {reader}")
            if candidates: mapping[(card,reader)] = candidates.pop()
    missing = [key for key in sorted(EXPECTED) if key not in mapping or not mapping[key].is_dir()]
    if missing:
        raise FileNotFoundError(f"Missing collections {missing}. Configure PINPAD_DATA_ROOT and collection_paths.csv; see README.")
    paths = [mapping[k] for k in sorted(EXPECTED)]
    if len(set(paths)) != len(paths): raise ValueError("Different physical collections map to the same directory")
    return mapping


def read_collection(key,mapping):
    parts = []
    for button in range(N_BUTTONS):
        hits = [p for p in (mapping[key]/f"button{button}.npy",mapping[key]/f"Button{button}.npy") if p.is_file()]
        if len(hits)!=1: raise FileNotFoundError(f"Expected one button{button}.npy in {mapping[key]}")
        x = np.load(hits[0],allow_pickle=False)
        if x.shape!=(N_PER_BUTTON,RESPONSE_LENGTH) or not np.iscomplexobj(x) or not np.isfinite(x).all():
            raise ValueError(f"Expected finite complex ({N_PER_BUTTON},{RESPONSE_LENGTH}) array: {hits[0]}")
        parts.append(x)
    return np.concatenate(parts),np.repeat(np.arange(N_BUTTONS),N_PER_BUTTON)


def load_group(keys,mapping):
    values = [read_collection(k,mapping) for k in keys]
    return (np.concatenate([x for x,y in values]),np.concatenate([y for x,y in values]),
            np.concatenate([np.full(len(y),k[0]) for k,(x,y) in zip(keys,values)]))


def features(raw):
    x = np.asarray(raw)
    if x.ndim!=2 or x.shape[1]!=RESPONSE_LENGTH or not np.iscomplexobj(x) or not np.isfinite(x).all():
        raise ValueError("Expected finite complex response segments")
    if DRIFT_MODE=="linear_complex":
        t = np.linspace(-1.,1.,x.shape[1])
        x = x-x.mean(axis=1,keepdims=True)
        slope = (x@t)/np.dot(t,t)
        x = x-slope[:,None]*t
    elif DRIFT_MODE!="none": raise ValueError("DRIFT_MODE must be none or linear_complex")
    if INPUT_FORMAT!="magnitude": raise ValueError("This release uses magnitude inputs")
    return np.abs(x).astype(np.float32)[...,None]


def normalize(x,stats):
    return ((x-np.asarray(stats['mean'],dtype=np.float32))/np.asarray(stats['std'],dtype=np.float32)).astype(np.float32)


def load_splits():
    f = pd.read_csv(SPLITS_CSV,dtype=str,keep_default_na=False)
    if not {"run_id","card","reader","partition"} <= set(f): raise ValueError("Invalid card split columns")
    f['card'] = f.card.astype(int)
    f['reader'] = f.reader.str.upper().str.strip()
    if set(f.run_id)!={f"run{i}" for i in range(1,6)}: raise ValueError("Provide exactly run1–run5")
    for run_id,g in f.groupby('run_id'):
        if g.duplicated(['card','reader']).any() or set(zip(g.card,g.reader))!=EXPECTED:
            raise ValueError(f"{run_id}: assign every development collection exactly once")
        if not set(g.partition)<={'train','validation'}: raise ValueError("Invalid development partition")
        source = g.reader.eq('R1') & g.card.between(1,5)
        if set(g.loc[source,'partition'])!={'train','validation'}: raise ValueError("Source needs train and validation cards")
        if not g.loc[~source,'partition'].eq('train').all(): raise ValueError("Targets must be unlabeled adaptation training collections")
    return f


def response_hashes(raw):
    return np.asarray([hashlib.sha256(np.ascontiguousarray(x,dtype='<c16').tobytes()).hexdigest() for x in raw])


def prepare_data(run_id,splits):
    mapping = collection_paths()
    raw,y,cards = load_group(SOURCE,mapping)
    selected = splits[splits.run_id.eq(run_id)]
    source = selected[selected.reader.eq('R1') & selected.card.between(1,5)]
    validation_cards = set(source.loc[source.partition.eq('validation'),'card'])
    val = np.isin(cards,list(validation_cards)); train = ~val
    h = response_hashes(raw)
    if set(h[train]) & set(h[val]): raise ValueError("Identical source captures cross training and validation")
    x = features(raw)
    stats = dict(mean=x[train].mean(axis=(0,1),dtype=np.float64).astype(np.float32).tolist(),
        std=np.maximum(x[train].std(axis=(0,1),dtype=np.float64).astype(np.float32),1e-7).tolist())
    x = normalize(x,stats)
    targets = {}
    audit = [pd.DataFrame(dict(domain='Source',card=cards,partition=np.where(val,'validation','train'),response_sha256=h))]
    for domain,keys in (("CC",ADAPT_CC),("CR",ADAPT_CR),("CCR",ADAPT_CCR)):
        target_raw,unused,tcards = load_group(keys,mapping)
        th = response_hashes(target_raw)
        if set(th)&set(h[val]): raise ValueError(f"{domain}: adaptation captures overlap source validation")
        targets[domain] = normalize(features(target_raw),stats)
        audit.append(pd.DataFrame(dict(domain=domain,card=tcards,partition='train',response_sha256=th)))
    audit = pd.concat(audit,ignore_index=True)
    prep = dict(input_format=INPUT_FORMAT,signal_length=RESPONSE_LENGTH,normalization=NORMALIZATION,
        source_stats=stats,drift_mode=DRIFT_MODE,classes=[f"B{i}" for i in range(12)])
    return x[train],y[train],x[val],y[val],targets,prep,audit


def save_json(path,data):
    Path(path).write_text(json.dumps(data,indent=2)+"\n")


def save_audit(run,selected,audit):
    selected.to_csv(run/"card_split.csv",index=False)
    audit.to_csv(run/"capture_hashes.csv",index=False)
    duplicates = audit[audit.response_sha256.duplicated(keep=False)]
    duplicates.to_csv(run/"duplicate_response_audit.csv",index=False)
    if not duplicates.empty: print("Repeated response hashes recorded in duplicate_response_audit.csv")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import sklearn, joblib, importlib.metadata
from adapt.feature_based import CORAL

if importlib.metadata.version("adapt")!="0.4.5":
    raise RuntimeError("Install requirements-rf-adapt.txt: this notebook targets adapt 0.4.5")

COMPONENTS = 64
EPS = 1e-5
PCA_KWARGS = dict(svd_solver="full",whiten=False)
RF_KWARGS = dict(criterion="gini",max_features="sqrt",min_samples_split=2,
    min_samples_leaf=1,max_depth=None,bootstrap=True,class_weight=None,n_jobs=-1)
SPLITS_CSV = Path(os.environ.get("PINPAD_RF_SPLITS",PROJECT/"manifests/rf_card_splits.csv"))


def common_target_view(target_z):
    arrays=[np.asarray(target_z[k],dtype=np.float64) for k in ("CC","CR","CCR")]
    if any(x.ndim!=2 or len(x)<2 for x in arrays): raise ValueError("Each target domain needs at least two rows")
    if len({x.shape[1] for x in arrays})!=1: raise ValueError("Target feature dimensions differ")
    means=[x.mean(axis=0) for x in arrays]
    common_mean=np.mean(means,axis=0)
    common_covariance=np.mean([np.cov(x,rowvar=False,ddof=1) for x in arrays],axis=0)
    total=sum(len(x) for x in arrays)
    reference=np.concatenate([(x-mu)*np.sqrt((total-1)/(3*(len(x)-1))) for x,mu in zip(arrays,means)])
    if not np.allclose(np.cov(reference,rowvar=False,ddof=1),common_covariance,rtol=1e-7,atol=1e-10):
        raise FloatingPointError("Common-target covariance preparation failed")
    return reference,common_mean,common_covariance


def align_with_adapt(source_z,target_z,epsilon,seed):
    source_z=np.asarray(source_z,dtype=np.float64)
    source_mean=source_z.mean(axis=0)
    reference,target_mean,target_covariance=common_target_view(target_z)
    alignment=CORAL(lambda_=epsilon,verbose=0,random_state=seed)
    centered_aligned=alignment.fit_transform(source_z-source_mean,reference)
    transform=alignment.transform(np.eye(source_z.shape[1]),domain="src")
    aligned=centered_aligned+target_mean
    identity=np.eye(source_z.shape[1])
    source_covariance=np.cov(source_z,rowvar=False,ddof=1)
    if not np.allclose(alignment.Cs_,source_covariance+epsilon*identity,rtol=1e-7,atol=1e-10):
        raise FloatingPointError("ADAPT source covariance differs from the manuscript formula")
    if not np.allclose(alignment.Ct_,target_covariance+epsilon*identity,rtol=1e-7,atol=1e-10):
        raise FloatingPointError("ADAPT target covariance differs from the manuscript formula")
    if not np.allclose(aligned,(source_z-source_mean)@transform+target_mean,rtol=1e-7,atol=1e-9):
        raise FloatingPointError("ADAPT alignment and exported transformation disagree")
    if not np.isfinite(aligned).all(): raise FloatingPointError("Nonfinite aligned features")
    return aligned,transform,source_mean,target_mean,source_covariance,target_covariance


In [ ]:
if any(v is None for v in (COMPONENTS,EPS,PCA_KWARGS,RF_KWARGS)):
    raise ValueError("Supply original PCA dimension, covariance epsilon, PCA_KWARGS, and RF_KWARGS")
if {'n_components','random_state'} & set(PCA_KWARGS): raise ValueError("Duplicate PCA settings")
if {'n_estimators','random_state'} & set(RF_KWARGS): raise ValueError("Duplicate RF settings")
splits = load_splits()
reference = None
for index in range(5):
    run_id,SEED = f"run{index+1}",RUN_SEEDS[index]
    selected = splits[splits.run_id.eq(run_id)]
    train_collections = set(zip(selected.loc[selected.partition.eq('train'),'card'],
                                selected.loc[selected.partition.eq('train'),'reader']))
    if reference is None: reference = train_collections
    if train_collections != reference:
        raise ValueError("RF section specifies the same source-training/adaptation data across runs; supply original RF run assignments")
    RUN = OUTPUT / "RF" / run_id
    if RUN.exists(): raise FileExistsError(f"Choose a new output root: {RUN}")
    X,y,Xv,yv,targets,preprocessing,audit = prepare_data(run_id,splits)
    source_features = X.reshape(len(X),-1)
    target_features = {k:v.reshape(len(v),-1) for k,v in targets.items()}
    if not 2 <= COMPONENTS <= min(source_features.shape):
        raise ValueError("Invalid PCA dimension")
    if EPS <= 0: raise ValueError("EPS must be positive")
    pca = PCA(n_components=COMPONENTS,random_state=SEED,**PCA_KWARGS)
    source_z = pca.fit_transform(source_features)
    target_z = {k:pca.transform(v) for k,v in target_features.items()}
    
    
    source_aligned,transform,mu_source,mu_target,c_source,c_target = align_with_adapt(
        source_z,target_z,EPS,SEED)
    rf_source = RandomForestClassifier(n_estimators=300,random_state=SEED,**RF_KWARGS).fit(source_z,y)
    rf = RandomForestClassifier(n_estimators=300,random_state=SEED,**RF_KWARGS).fit(source_aligned,y)
    RUN.mkdir(parents=True)
    save_audit(RUN,selected,audit)
    joblib.dump(dict(pca=pca,rf=rf,rf_source=rf_source,mu_s=mu_source,mu_t=mu_target,
                    transform=transform,preprocessing=preprocessing,
                    source_covariance=c_source,target_common_covariance=c_target,
                    alignment_library={"name":"adapt","version":importlib.metadata.version("adapt"),
                    "method":"CORAL.fit_transform; explicit source centering and common-target mean restoration"}),RUN/"RF_models.joblib")
    save_json(RUN/"preprocessing.json",preprocessing)
    save_json(RUN/"training_config.json",dict(seed=SEED,pca_components=COMPONENTS,
        covariance_epsilon=EPS,n_estimators=300,adapt=importlib.metadata.version("adapt"),
        alignment="ADAPT CORAL with equal-domain covariance preparation and mean restoration",protocol="reference RF repeated seeds; not original card-based evaluations",python=platform.python_version(),pandas=pd.__version__,joblib=joblib.__version__,pca_kwargs=PCA_KWARGS,rf_kwargs=RF_KWARGS,scikit_learn=sklearn.__version__,numpy=np.__version__))
    print("Saved:",RUN)


In [ ]:
def preprocess_new_responses(raw,prep):
    x = np.asarray(raw)
    if x.ndim==1: x=x[None,:]
    if x.ndim!=2 or x.shape[1]!=prep['signal_length'] or not np.iscomplexobj(x) or not np.isfinite(x).all():
        raise ValueError("Expected finite complex responses (N,T)")
    if prep['drift_mode']=='linear_complex':
        t=np.linspace(-1.,1.,x.shape[1]); x=x-x.mean(axis=1,keepdims=True)
        x=x-((x@t)/np.dot(t,t))[:,None]*t
    elif prep['drift_mode']!='none': raise ValueError("Unsupported saved drift mode")
    mag=np.abs(x).astype(np.float32)[...,None]
    stats=prep['source_stats']
    return ((mag-np.asarray(stats['mean'],dtype=np.float32))/np.asarray(stats['std'],dtype=np.float32)).astype(np.float32)


def load_rf_predictor(run_directory):
    saved=joblib.load(Path(run_directory)/"RF_models.joblib")
    def predict(raw):
        x=preprocess_new_responses(raw,saved['preprocessing'])
        z=saved['pca'].transform(x.reshape(len(x),-1))
        return saved['rf'].predict_proba(z)
    return predict
